# Nykaa Domain Support Agent — Task 15: SQLite-Based LangGraph Checkpointing

## Objective

Task 15 requires the existing four-node LangGraph support agent to use a persistent SQLite checkpointer (`checkpoints.sqlite`) keyed by a thread ID.

The demonstration must prove:

1. The graph executes at least two nodes.
2. Execution is deliberately paused before the remaining node(s).
3. The same thread ID is resumed.
4. Previously completed node results are loaded from the checkpoint and those nodes are **not executed again**.

This notebook follows the four-node Task 7 graph:
- `classify_intent`
- `retrieve_policy`
- `lookup_order`
- `compose_response`

It uses an order-status query for the checkpoint demonstration. The expected path is:

`START → classify_intent → lookup_order → [PAUSE] → compose_response → END`

The policy-retrieval node remains part of the graph as the alternative conditional branch.

**Run this notebook from `C:\Users\singh\Downloads` (your Nykaa project folder), where `chroma_db/` and `order_lookup.py` are available.**

## 1. Install the SQLite checkpointer package

In [ ]:
%pip install -U langgraph langgraph-checkpoint-sqlite chromadb sentence-transformers

## 2. Imports and project configuration

In [ ]:
from pathlib import Path
from typing import TypedDict, Optional, Literal
import re
import json
import sqlite3
import uuid

import chromadb
from sentence_transformers import SentenceTransformer

from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.sqlite import SqliteSaver

PROJECT_DIR = Path.cwd()
CHROMA_DIR = PROJECT_DIR / "chroma_db"
DB_PATH = PROJECT_DIR / "checkpoints.sqlite"
EVIDENCE_DIR = PROJECT_DIR / "results"
EVIDENCE_DIR.mkdir(exist_ok=True)

print("Project directory:", PROJECT_DIR.resolve())
print("ChromaDB directory:", CHROMA_DIR.resolve())
print("SQLite checkpoint database:", DB_PATH.resolve())

if not CHROMA_DIR.exists():
    raise FileNotFoundError(
        f"Missing {CHROMA_DIR}. Run the Task 3 indexing notebook in this project folder."
    )

if not (PROJECT_DIR / "order_lookup.py").exists():
    raise FileNotFoundError(
        "order_lookup.py is missing. Complete the Task 14 integration step first."
    )


## 3. Load the existing Task 5 RAG collection and Task 6 order lookup

This reuses the same sentence-based collection and the `check_order_status()` function exposed through MCP in Task 14. The escalation formula is not reimplemented here.

In [ ]:
from order_lookup import check_order_status

COLLECTION_NAME = "nykaa_sentence_chunks"
MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
TOP_K = 3

chroma_client = chromadb.PersistentClient(path=str(CHROMA_DIR))
policy_collection = chroma_client.get_collection(COLLECTION_NAME)
embedding_model = SentenceTransformer(MODEL_NAME)

print("RAG collection:", COLLECTION_NAME)
print("Indexed chunks:", policy_collection.count())
print("Order lookup test:", check_order_status("ORD1001"))

assert policy_collection.count() > 0
assert check_order_status("ORD1001").get("found") is True
print("PASS: Existing RAG index and Task 6 order lookup are available.")


## 4. Define the shared LangGraph state

In [ ]:
class SupportState(TypedDict, total=False):
    query: str
    intent: Literal["policy", "order"]
    record_id: Optional[str]
    retrieved_context: list
    order_result: dict
    answer: str
    sources: list
    route_taken: str

print("PASS: SupportState defined.")


## 5. Define the four Task 7 nodes

A small execution log is added for the demonstration. It records when each node function actually runs. This lets us prove that `classify_intent` and `lookup_order` are not called again after resuming the checkpoint.

In [ ]:
node_execution_log = []

ORDER_ID_PATTERN = re.compile(r"\bORD\d{4,}\b", flags=re.IGNORECASE)
ORDER_INTENT_KEYWORDS = {
    "order status", "track my order", "track order",
    "where is my order", "check my order", "order update"
}

def classify_intent(state: SupportState) -> dict:
    node_execution_log.append("classify_intent")
    query = state["query"].strip()
    query_lower = query.lower()

    id_match = ORDER_ID_PATTERN.search(query)
    has_order_language = any(term in query_lower for term in ORDER_INTENT_KEYWORDS)

    if id_match or has_order_language:
        intent = "order"
        record_id = id_match.group(0).upper() if id_match else None
    else:
        intent = "policy"
        record_id = None

    print(f"[EXECUTED] classify_intent: intent={intent}, record_id={record_id}")
    return {
        "intent": intent,
        "record_id": record_id,
        "route_taken": intent
    }


def retrieve_policy(state: SupportState) -> dict:
    node_execution_log.append("retrieve_policy")
    query_embedding = embedding_model.encode(
        [state["query"]],
        normalize_embeddings=True
    )[0].tolist()

    result = policy_collection.query(
        query_embeddings=[query_embedding],
        n_results=min(TOP_K, policy_collection.count()),
        include=["documents", "metadatas", "distances"]
    )

    documents = result.get("documents", [[]])[0]
    metadatas = result.get("metadatas", [[]])[0]
    distances = result.get("distances", [[]])[0]

    contexts = []
    for document, metadata, distance in zip(documents, metadatas, distances):
        metadata = metadata or {}
        contexts.append({
            "text": document,
            "source": metadata.get("source", "unknown"),
            "similarity": round(1 - float(distance), 4)
        })

    print(f"[EXECUTED] retrieve_policy: retrieved {len(contexts)} chunks")
    return {"retrieved_context": contexts}


def lookup_order(state: SupportState) -> dict:
    node_execution_log.append("lookup_order")
    record_id = state.get("record_id")

    if not record_id:
        result = {
            "found": False,
            "error": "Please provide your order ID (for example, ORD1001)."
        }
    else:
        result = check_order_status(record_id)

    print("[EXECUTED] lookup_order: found =", result.get("found"))
    return {"order_result": result}


def compose_response(state: SupportState) -> dict:
    node_execution_log.append("compose_response")

    if state.get("intent") == "order":
        result = state.get("order_result", {})
        if not result.get("found"):
            answer = result.get("error", "I could not find that order.")
            sources = []
        else:
            escalation_text = (
                "This order is recommended for human support review."
                if result["requires_escalation"]
                else "This order does not currently meet the project's escalation threshold."
            )
            answer = (
                f"Order {result['record_id']} has status: {result['status']}. "
                f"Order value: ₹{result['order_value_inr']:,.2f}. "
                f"Escalation score: {result['escalation_score']:.2f}. "
                f"{escalation_text}"
            )
            sources = ["orders dataset (synthetic project data)"]
    else:
        contexts = state.get("retrieved_context", [])
        useful = [
            item for item in contexts
            if item.get("text") and item.get("source") != "unknown"
        ]
        if not useful:
            answer = "I don't know based on the available policy documents."
            sources = []
        else:
            answer = (
                "According to the available project policy documents:\n\n"
                + "\n\n".join(
                    f"[{item['source']}] {item['text']}" for item in useful
                )
            )
            sources = list(dict.fromkeys(item["source"] for item in useful))

    print("[EXECUTED] compose_response: final answer created")
    return {"answer": answer, "sources": sources}

print("PASS: All four Task 7 node functions are defined.")


## 6. Build the same four-node conditional LangGraph

In [ ]:
def route_after_classification(state: SupportState) -> Literal["policy", "order"]:
    return "order" if state.get("intent") == "order" else "policy"

builder = StateGraph(SupportState)

builder.add_node("classify_intent", classify_intent)
builder.add_node("retrieve_policy", retrieve_policy)
builder.add_node("lookup_order", lookup_order)
builder.add_node("compose_response", compose_response)

builder.add_edge(START, "classify_intent")
builder.add_conditional_edges(
    "classify_intent",
    route_after_classification,
    {
        "policy": "retrieve_policy",
        "order": "lookup_order"
    }
)
builder.add_edge("retrieve_policy", "compose_response")
builder.add_edge("lookup_order", "compose_response")
builder.add_edge("compose_response", END)

print("PASS: Four-node conditional graph built.")
print(builder.compile().get_graph().draw_mermaid())


## 7. Configure SQLite checkpointing and pause before the final node

`SqliteSaver` persists graph state to `checkpoints.sqlite`. The `thread_id` identifies this particular conversation/run.

`interrupt_before=["compose_response"]` tells LangGraph to pause immediately before the final response node. By that point, the order query has already passed through `classify_intent` and `lookup_order`.

In [ ]:
THREAD_ID = "nykaa_task15_checkpoint_demo_001"
config = {
    "configurable": {
        "thread_id": THREAD_ID
    }
}

checkpoint_db = SqliteSaver.from_conn_string(str(DB_PATH))
checkpointer = checkpoint_db.__enter__()

checkpointed_agent = builder.compile(
    checkpointer=checkpointer,
    interrupt_before=["compose_response"]
)

print("SQLite checkpointing enabled.")
print("Database:", DB_PATH.resolve())
print("Thread ID:", THREAD_ID)
print("Graph will pause before: compose_response")


## 8. Start the graph and deliberately interrupt it

The order-status request should execute:
1. `classify_intent`
2. `lookup_order`

It should then pause before `compose_response`. The state produced by the first two nodes is saved to SQLite.

In [ ]:
node_execution_log.clear()

initial_input = {
    "query": "Please check the status of order ORD1001."
}

print("STARTING GRAPH")
partial_result = checkpointed_agent.invoke(initial_input, config)

print("\nNODE EXECUTION LOG AFTER INTERRUPT:")
print(node_execution_log)

snapshot_before_resume = checkpointed_agent.get_state(config)

print("\nCHECKPOINT SNAPSHOT:")
print("Thread ID:", snapshot_before_resume.config["configurable"]["thread_id"])
print("Next node(s):", snapshot_before_resume.next)
print("Saved state keys:", list(snapshot_before_resume.values.keys()))
print("Saved intent:", snapshot_before_resume.values.get("intent"))
print("Saved order result:", snapshot_before_resume.values.get("order_result"))

assert node_execution_log == ["classify_intent", "lookup_order"], (
    "Expected exactly classify_intent and lookup_order to run before the pause."
)
assert snapshot_before_resume.next == ("compose_response",), (
    f"Expected pause before compose_response, got: {snapshot_before_resume.next}"
)
assert snapshot_before_resume.values.get("order_result", {}).get("found") is True

print("\nPASS: Two nodes executed and their results were checkpointed.")
print("PASS: Execution paused before compose_response.")


## 9. Resume the SAME thread ID from the SQLite checkpoint

Calling `invoke(None, config)` resumes the paused graph. Do not pass the original input again. LangGraph loads the saved state for this same `thread_id` and continues from `compose_response`.

In [ ]:
print("RESUMING GRAPH USING THE SAME THREAD ID:")
print(THREAD_ID)

resumed_result = checkpointed_agent.invoke(None, config)

print("\nNODE EXECUTION LOG AFTER RESUME:")
print(node_execution_log)

print("\nFINAL ANSWER:")
print(resumed_result.get("answer"))

assert node_execution_log == [
    "classify_intent",
    "lookup_order",
    "compose_response"
], (
    "A completed node ran again, or the final node did not execute exactly once."
)

assert resumed_result.get("answer")
assert resumed_result.get("order_result", {}).get("record_id") == "ORD1001"

print("\nPASS: Graph resumed and completed.")
print("PASS: classify_intent and lookup_order were NOT re-executed.")
print("PASS: Only compose_response executed after resume.")


## 10. Verify the completed checkpoint and SQLite database

In [ ]:
final_snapshot = checkpointed_agent.get_state(config)

print("FINAL CHECKPOINT:")
print("Thread ID:", final_snapshot.config["configurable"]["thread_id"])
print("Next node(s):", final_snapshot.next)
print("Final answer stored:", bool(final_snapshot.values.get("answer")))
print("Execution log:", node_execution_log)

assert final_snapshot.next == ()
assert final_snapshot.values.get("answer")
assert DB_PATH.exists() and DB_PATH.stat().st_size > 0

# Confirm the SQLite database can be opened and contains checkpoint tables.
with sqlite3.connect(DB_PATH) as conn:
    tables = conn.execute(
        "SELECT name FROM sqlite_master WHERE type='table' ORDER BY name"
    ).fetchall()

print("\nSQLite tables:", [row[0] for row in tables])
assert tables, "No tables found in the SQLite checkpoint database."

print("PASS: Final graph state is complete.")
print("PASS: SQLite checkpoint database exists and contains tables.")


## 11. Save Task 15 evidence and close the SQLite connection

In [ ]:
evidence = {
    "task": "Task 15 — SQLite-based LangGraph checkpointing",
    "database": str(DB_PATH.resolve()),
    "thread_id": THREAD_ID,
    "graph_nodes": [
        "classify_intent",
        "retrieve_policy",
        "lookup_order",
        "compose_response"
    ],
    "interrupt_before": "compose_response",
    "nodes_executed_before_interrupt": [
        "classify_intent",
        "lookup_order"
    ],
    "checkpointed_state_before_resume": {
        "intent": snapshot_before_resume.values.get("intent"),
        "record_id": snapshot_before_resume.values.get("record_id"),
        "order_result": snapshot_before_resume.values.get("order_result")
    },
    "nodes_executed_after_resume_total": node_execution_log,
    "completed_nodes_reexecuted": False,
    "final_answer": resumed_result.get("answer"),
    "final_state_complete": final_snapshot.next == (),
    "sqlite_tables": [row[0] for row in tables]
}

evidence_path = EVIDENCE_DIR / "task15_checkpoint_evidence.json"
evidence_path.write_text(
    json.dumps(evidence, indent=2, ensure_ascii=False, default=str),
    encoding="utf-8"
)

print("Evidence saved:", evidence_path.resolve())
print("PASS: Task 15 checkpoint evidence saved.")

# Close the SqliteSaver connection after all graph operations are complete.
checkpoint_db.__exit__(None, None, None)
print("SQLite checkpointer connection closed.")


## 12. Task 15 acceptance checklist

Task 15 is complete when the notebook demonstrates:

- [ ] `langgraph-checkpoint-sqlite` installed.
- [ ] Four-node Task 7 graph compiled with a SQLite checkpointer.
- [ ] `checkpoints.sqlite` created.
- [ ] A stable `thread_id` used for the run.
- [ ] `classify_intent` and `lookup_order` executed before interruption.
- [ ] Graph deliberately paused before `compose_response`.
- [ ] Saved state and pending node shown from `get_state()`.
- [ ] Same `thread_id` resumed using `invoke(None, config)`.
- [ ] Execution log proves earlier nodes were not re-executed.
- [ ] Final graph completed and evidence JSON saved.

**Expected execution log after resume:**

```text
[
  'classify_intent',
  'lookup_order',
  'compose_response'
]
```

The first two nodes appear only once. This is the central proof that their results were recovered from the SQLite checkpoint.

**Important:** Keep `checkpoints.sqlite` and `results/task15_checkpoint_evidence.json` in your project repository. Do not delete the database before capturing your Task 15 evidence.